# Wine Quality データセットを用いた機械学習入門

本ノートブックでは、UCI Machine Learning Repository の Wine Quality データセットを使って以下の3つの機械学習手法を実習します。

1. **一般化線形モデル (GLM)** — ポアソン回帰による品質スコアの予測
2. **ランダムフォレスト** — 品質の予測と特徴量重要度の可視化
3. **階層的クラスタリング** — ラベルなしで赤ワインと白ワインを区別できるかの検証

## データセットについて

ポルトガルの「Vinho Verde」ワインについて、赤ワイン約1,600本・白ワイン約4,900本の以下の情報が記録されています。

| 列名 | 意味 |
| --- | --- |
| `fixed acidity` | 不揮発性酸度 |
| `volatile acidity` | 揮発性酸度(高いと酢っぽくなる) |
| `citric acid` | クエン酸 |
| `residual sugar` | 残糖 |
| `chlorides` | 塩化物 |
| `free sulfur dioxide` | 遊離亜硫酸 |
| `total sulfur dioxide` | 総亜硫酸 |
| `density` | 密度 |
| `pH` | pH |
| `sulphates` | 硫酸塩 |
| `alcohol` | アルコール度数 |
| `quality` | 品質スコア(0〜10の整数、官能評価) |

## 0. 必要ライブラリのインストール(初回のみ)

In [ ]:
# 初回のみ実行してください
# !pip install scikit-learn seaborn matplotlib pandas scipy statsmodels

## 1. データの取得と確認

UCI のサーバから直接 CSV を読み込みます。赤ワインと白ワインの2ファイルを取得し、`color` 列を付けて結合します。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

URL_RED = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
URL_WHITE = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"

red = pd.read_csv(URL_RED, sep=";")
white = pd.read_csv(URL_WHITE, sep=";")

# 色のラベルを付与して結合
red["color"] = "red"
white["color"] = "white"
df = pd.concat([red, white], ignore_index=True)

print(f"赤ワイン: {len(red)} 本")
print(f"白ワイン: {len(white)} 本")
print(f"合計    : {len(df)} 本")
df.head()

In [ ]:
# 列名にスペースが含まれていると扱いにくいので、アンダースコアに置換
df.columns = df.columns.str.replace(" ", "_")
print("カラム名:", df.columns.tolist())

In [ ]:
# 各列の型と欠損数
df.info()

In [ ]:
# 基本統計量
df.describe()

In [ ]:
# 品質スコアの分布
print("品質スコアの分布:")
print(df["quality"].value_counts().sort_index())

## 2. 探索的データ解析(EDA)

まずは品質スコアの分布と、特徴量との関係を可視化します。

In [ ]:
sns.set_style("whitegrid")

# 品質スコアのヒストグラム(赤白別)
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, x="quality", hue="color",
              palette={"red": "#b22222", "white": "#daa520"}, ax=ax)
ax.set_title("品質スコアの分布(赤ワイン vs 白ワイン)")
plt.tight_layout()
plt.show()

In [ ]:
# 品質と各特徴量の関係を箱ひげ図で
features = [c for c in df.columns if c not in ["quality", "color"]]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, feat in zip(axes.flat, features):
    sns.boxplot(data=df, x="quality", y=feat, ax=ax,
                color="lightsteelblue", fliersize=2)
    ax.set_title(feat)
# 余ったサブプロットを非表示に
for ax in axes.flat[len(features):]:
    ax.set_visible(False)
plt.suptitle("品質スコアと各特徴量の関係", y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# 相関行列のヒートマップ
plt.figure(figsize=(10, 8))
corr = df[features + ["quality"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, cbar_kws={"shrink": 0.8})
plt.title("特徴量間の相関行列")
plt.tight_layout()
plt.show()

**観察ポイント**
- アルコール度数と品質に正の相関がある(箱ひげ図でも右肩上がり)
- 揮発性酸度と品質には負の相関がある(高いと「酢っぽい」のでマイナス評価)
- `density` は `alcohol` や `residual_sugar` と強く相関しており、多重共線性に注意

## 3. 一般化線形モデル(ポアソン回帰)で品質を予測

品質スコアは 0〜10 の整数値(計数データ)なので、ポアソン回帰が自然な選択です。`statsmodels` を使うと係数の p 値や信頼区間が見えるので、教育用途に向いています。

In [ ]:
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

X = df[features].copy()
y = df["quality"].copy()

# 標準化(係数の大小比較がしやすくなる)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features)

# 訓練・テスト分割
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

# ポアソン回帰(GLM: family=Poisson, link=log)
X_train_sm = sm.add_constant(X_train)
glm = sm.GLM(y_train, X_train_sm, family=sm.families.Poisson())
result = glm.fit()
print(result.summary())

In [ ]:
# テストデータでの予測
X_test_sm = sm.add_constant(X_test)
y_pred = result.predict(X_test_sm)

print(f"MAE  (平均絶対誤差): {mean_absolute_error(y_test, y_pred):.3f}")
print(f"RMSE (二乗平均平方根誤差): {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")

In [ ]:
# 係数の可視化(標準化済みなので大小比較が可能)
coefs = result.params.drop("const").sort_values()

plt.figure(figsize=(8, 5))
colors = ["crimson" if v < 0 else "steelblue" for v in coefs]
coefs.plot(kind="barh", color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("係数(標準化後)")
plt.title("ポアソン回帰の係数(青=品質に正の効果、赤=負の効果)")
plt.tight_layout()
plt.show()

**学生への質問**
- 品質を高めるのに最も効く特徴量は何ですか?
- 揮発性酸度(volatile_acidity)の係数の符号は、ワインの常識的な評価と整合していますか?
- p 値の大きい(有意でない)変数はどれですか? それは何を意味する?

## 4. ランダムフォレストで品質を予測

非線形な関係や相互作用を捉えられます。GLM との精度比較がポイントです。

In [ ]:
from sklearn.ensemble import RandomForestRegressor

X = df[features]
y = df["quality"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print(f"ランダムフォレスト")
print(f"  MAE : {mean_absolute_error(y_test, y_pred_rf):.3f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.3f}")

# GLM との比較表
comparison = pd.DataFrame({
    "モデル": ["GLM (Poisson)", "Random Forest"],
    "MAE":  [mean_absolute_error(y_test, y_pred), mean_absolute_error(y_test, y_pred_rf)],
    "RMSE": [np.sqrt(mean_squared_error(y_test, y_pred)), np.sqrt(mean_squared_error(y_test, y_pred_rf))],
})
print("\nモデル比較:")
print(comparison.to_string(index=False))

In [ ]:
# 特徴量重要度の可視化
importance = pd.Series(rf.feature_importances_, index=features).sort_values()

plt.figure(figsize=(8, 5))
importance.plot(kind="barh", color="darkgreen")
plt.xlabel("Feature Importance")
plt.title("ランダムフォレストによる特徴量重要度")
plt.tight_layout()
plt.show()

In [ ]:
# 予測値 vs 実測値の散布図
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, pred, title in [(axes[0], y_pred, "GLM (Poisson)"),
                         (axes[1], y_pred_rf, "Random Forest")]:
    ax.scatter(y_test, pred, alpha=0.3, s=15)
    ax.plot([3, 9], [3, 9], "r--", linewidth=1)
    ax.set_xlabel("実測値 (quality)")
    ax.set_ylabel("予測値")
    ax.set_title(title)
    ax.set_xlim(2.5, 9.5)
    ax.set_ylim(2.5, 9.5)

plt.tight_layout()
plt.show()

**学生への質問**
- GLM とランダムフォレストでは、どちらが精度が高い? その差はどれくらい?
- GLM の係数とランダムフォレストの重要度は、上位の変数が一致している? 違うとしたらなぜ?
- 予測値 vs 実測値のプロットを見ると、両モデルとも極端な値(3 や 8、9)の予測が苦手な様子が見える。なぜか?

## 5. 階層的クラスタリング(教師なし学習)

色ラベル(red/white)を **一切使わず** に、化学成分だけから赤・白を区別できるかを試します。

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# 計算量を抑えるため、ランダムに800サンプル抽出してクラスタリング
sample = df.sample(n=800, random_state=42).reset_index(drop=True)

X_sample = sample[features]
X_scaled_sample = StandardScaler().fit_transform(X_sample)

# Ward 法でリンケージを計算
Z = linkage(X_scaled_sample, method="ward")

# デンドログラム
plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode="lastp", p=30, leaf_rotation=90, leaf_font_size=10)
plt.title("階層的クラスタリングのデンドログラム (Ward法、800サンプル)")
plt.xlabel("サンプル(またはクラスタ)")
plt.ylabel("距離")
plt.axhline(y=40, color="red", linestyle="--", label="2クラスタで切る位置")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 2クラスタに切り分け
clusters = fcluster(Z, t=2, criterion="maxclust")
sample["cluster"] = clusters

# 答え合わせ:実際の色ラベルとの対応
print("クラスタ vs 色(答え合わせ):")
print(pd.crosstab(sample["cluster"], sample["color"]))

# 一致率を計算
ct = pd.crosstab(sample["cluster"], sample["color"])
best_match = ct.values.max(axis=1).sum() / ct.values.sum()
print(f"\nクラスタを色に最適にマッチさせた場合の一致率: {best_match:.1%}")

In [ ]:
# 可視化:総亜硫酸 × 揮発性酸度 の平面で「クラスタ」と「本当の色」を比較
# (赤白を最もよく分ける2変数の組み合わせ)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(
    data=sample, x="total_sulfur_dioxide", y="volatile_acidity",
    hue="cluster", palette="Set1", ax=axes[0], alpha=0.6, s=25
)
axes[0].set_title("クラスタリング結果(ラベル未使用)")

sns.scatterplot(
    data=sample, x="total_sulfur_dioxide", y="volatile_acidity",
    hue="color", palette={"red": "#b22222", "white": "#daa520"},
    ax=axes[1], alpha=0.6, s=25
)
axes[1].set_title("本当の色ラベル(答え)")

plt.tight_layout()
plt.show()

**学生への気づきポイント**

色ラベルを一切使っていないのに、赤ワインと白ワインがほぼ正確に分かれることが実感できます。これは、赤と白で化学成分(特に総亜硫酸と揮発性酸度)が大きく異なるためです。実際の醸造プロセスの違いがデータに反映されていることがわかります。

## 6. 実習のまとめとディスカッション課題

1. **回帰タスクと分類タスクの違い**
   - 品質スコア(0〜10)は連続値ではなく整数だが、なぜポアソン回帰を使う?
   - 通常の線形回帰(正規分布)を使うとどう変わる?(`family=sm.families.Gaussian()` で試してみよう)

2. **モデルの解釈性 vs 精度**
   - GLM は係数で「どの変数がどれくらい効くか」がわかるが、精度ではランダムフォレストに劣ることが多い。実務ではどちらを選ぶ?
   - 「ワインの品質を上げたい醸造家」と「品質を当てる予測モデルがほしいバイヤー」では選ぶモデルが違うかもしれない。

3. **多重共線性**
   - 相関行列で `density` が `alcohol` や `residual_sugar` と強く相関していた。これが GLM の係数推定にどう影響する?
   - ランダムフォレストは多重共線性に強いと言われる。なぜ?

4. **クラスタリングの解釈**
   - 赤白の区別は化学的に明らかなので、クラスタリングがうまくいった。では、品質スコアでクラスタリングしたらきれいに分かれる?(試してみよう)
   - クラスタ数を 3 や 4 にしたら、どんな解釈ができる?

## 発展課題

- **二値分類**: 品質 ≥ 7 を「良いワイン」とラベル付けし、ロジスティック回帰で予測してみる
- **赤白別の解析**: 赤ワインと白ワインで「品質に効く変数」が違うかを別々に GLM で確認する
- **ハイパーパラメータ調整**: ランダムフォレストの `n_estimators`, `max_depth`, `min_samples_leaf` を変えて精度の変化を確認する
- **正則化**: `sklearn.linear_model.Ridge` や `Lasso` を使って多重共線性に対処する
- **次元削減**: 主成分分析(PCA)で2次元に圧縮し、赤白がどう分布しているか可視化する